##### Copyright 2019 The TensorFlow Authors (https://www.tensorflow.org/tutorials/generative/deepdream)

# DeepDream - Implementação Mínima com Imagem de Felino

**FRA - Aula 23 - Prática DeepDream**

---

Este notebook implementa um exemplo mínimo de **DeepDream** usando uma imagem de um felino da Wikipedia e a arquitetura InceptionV3, conforme visto na aula prática.

O DeepDream foi introduzido pelo Google em 2015 como uma forma de visualizar o que as redes neurais "aprendem". A técnica funciona maximizando as ativações de camadas específicas da rede, criando padrões surreais e oníricos na imagem.

**Referência:** [Inceptionism: Going Deeper into Neural Networks](https://ai.googleblog.com/2015/06/inceptionism-going-deeper-into-neural.html)

---
## 1. Importação das Bibliotecas

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib as mpl
import IPython.display as display
import PIL.Image

print(f"TensorFlow version: {tf.__version__}")

---
## 2. Importação da Imagem

### Escolha da imagem para "sonhar"

Utilizaremos uma imagem de um felino (gato na neve) da Wikipedia, conforme especificado no exercício.

In [ ]:
# URL da imagem conforme instrução do exercício
url = "https://commons.wikimedia.org/wiki/Special:FilePath/Felis_catus-cat_on_snow.jpg"

In [ ]:
# Download da imagem e gravação em array Numpy
def download(url, max_dim=None):
    name = url.split('/')[-1]
    image_path = tf.keras.utils.get_file(name, origin=url)
    img = PIL.Image.open(image_path)
    if max_dim:
        img.thumbnail((max_dim, max_dim))
    return np.array(img)

# Normalização da imagem (desnormaliza de [-1, 1] para [0, 255])
def deprocess(img):
    img = 255 * (img + 1.0) / 2.0
    return tf.cast(img, tf.uint8)

# Exibe a imagem
def show(img):
    display.display(PIL.Image.fromarray(np.array(img)))

In [ ]:
# Redução do tamanho da imagem para facilitar o processamento
original_img = download(url, max_dim=500)
show(original_img)
display.display(display.HTML('Imagem cc-by: <a href="https://commons.wikimedia.org/wiki/File:Felis_catus-cat_on_snow.jpg">Von.grzanka</a>'))

---
## 3. Preparar o Modelo de Extração de Recursos

Faremos o download do modelo de classificação de imagens pré-treinado **InceptionV3**, que é semelhante ao modelo originalmente usado no DeepDream. 

A ideia no DeepDream é escolher uma camada (ou camadas) e **maximizar a "perda"** de forma que a imagem cada vez mais "excite" as camadas selecionadas.

In [ ]:
# Download do modelo InceptionV3 pré-treinado no ImageNet
base_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet')

In [ ]:
# Maximizando as ativações das camadas escolhidas
# mixed3 e mixed5 são camadas intermediárias que capturam padrões em diferentes níveis
names = ['mixed3', 'mixed5']
layers = [base_model.get_layer(name).output for name in names]

# Criação do modelo de sonho (dream model)
dream_model = tf.keras.Model(inputs=base_model.input, outputs=layers)

---
## 4. Cálculo da Perda (*Loss*)

A perda é a **soma das ativações** nas camadas escolhidas. Quanto maior a ativação, mais o modelo "reconhece" padrões na imagem.

In [ ]:
def calc_loss(img, model):
    """Calcula a perda como soma das médias das ativações das camadas."""
    # Converte a imagem em um batch de tamanho 1
    img_batch = tf.expand_dims(img, axis=0)
    layer_activations = model(img_batch)
    
    if len(layer_activations) == 1:
        layer_activations = [layer_activations]

    losses = []
    for act in layer_activations:
        loss = tf.math.reduce_mean(act)
        losses.append(loss)

    return tf.reduce_sum(losses)

---
## 5. Subida de Gradiente (*Gradient Ascent*)

Após calcular a perda para as camadas escolhidas, calculamos os **gradientes em relação à imagem** e os adicionamos à imagem original. Isso faz com que a imagem seja modificada para aumentar as ativações nas camadas selecionadas.

In [ ]:
class DeepDream(tf.Module):
    """Classe para executar o algoritmo DeepDream."""
    
    def __init__(self, model):
        self.model = model

    @tf.function(
        input_signature=(
            tf.TensorSpec(shape=[None, None, 3], dtype=tf.float32),
            tf.TensorSpec(shape=[], dtype=tf.int32),
            tf.TensorSpec(shape=[], dtype=tf.float32),
        )
    )
    def __call__(self, img, steps, step_size):
        print("Tracing")
        loss = tf.constant(0.0)

        for n in tf.range(steps):
            with tf.GradientTape() as tape:
                # Gradientes relativos à imagem
                tape.watch(img)
                loss = calc_loss(img, self.model)

            # Cálculo do gradiente da perda em relação aos pixels da imagem
            gradients = tape.gradient(loss, img)

            # Normalização dos gradientes
            gradients /= tf.math.reduce_std(gradients) + 1e-8

            # Na subida de gradiente, a "perda" é maximizada
            # Atualiza a imagem adicionando os gradientes
            img = img + gradients * step_size
            img = tf.clip_by_value(img, -1, 1)

        return loss, img

In [ ]:
# Instancia o objeto DeepDream
deepdream = DeepDream(dream_model)

---
## 6. Circuito Principal (*Main Loop*)

O Main Loop aplica o algoritmo DeepDream em uma **única escala** da imagem, executando múltiplos passos de subida de gradiente.

In [ ]:
def run_deep_dream_simple(img, steps=100, step_size=0.01):
    """Executa o DeepDream usando apenas o Main Loop (sem oitavas)."""
    
    # Pré-processamento da imagem para o InceptionV3
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    img = tf.convert_to_tensor(img)
    step_size = tf.convert_to_tensor(step_size)
    steps_remaining = steps
    step = 0
    
    while steps_remaining:
        if steps_remaining > 100:
            run_steps = tf.constant(100)
        else:
            run_steps = tf.constant(steps_remaining)
        steps_remaining -= run_steps
        step += run_steps

        loss, img = deepdream(img, run_steps, tf.constant(step_size))

        display.clear_output(wait=True)
        show(deprocess(img))
        print(f"Step {step}, loss {loss:.4f}")

    result = deprocess(img)
    display.clear_output(wait=True)
    show(result)

    return result

In [ ]:
# Executa o DeepDream com Main Loop
dream_img_mainloop = run_deep_dream_simple(img=original_img, steps=100, step_size=0.01)

---
## 7. Levando o Modelo até uma Oitava

A primeira tentativa com o Main Loop gera uma imagem, porém há alguns problemas:

1. A saída é **ruidosa** (isso pode ser resolvido com uma perda `tf.image.total_variation`)
2. A imagem é de **baixa resolução**
3. Os padrões parecem estar acontecendo na **mesma granularidade**

Para resolver esses problemas, aplicamos o DeepDream em **múltiplas escalas (oitavas)**. Isso permite que padrões em diferentes níveis de detalhe sejam realçados.

In [ ]:
import time

start = time.time()

# Fator de escala entre oitavas
OCTAVE_SCALE = 1.30

img = tf.constant(np.array(original_img))
base_shape = tf.shape(img)[:-1]
float_base_shape = tf.cast(base_shape, tf.float32)

# Processa de oitavas menores para maiores (-2 a 2 = 5 oitavas)
for n in range(-2, 3):
    new_shape = tf.cast(float_base_shape * (OCTAVE_SCALE ** n), tf.int32)
    img = tf.image.resize(img, new_shape).numpy()
    img = run_deep_dream_simple(img=img, steps=50, step_size=0.01)

display.clear_output(wait=True)

# Redimensiona para o tamanho original
img = tf.image.resize(img, base_shape)
img = tf.image.convert_image_dtype(img / 255.0, dtype=tf.uint8)
show(img)

end = time.time()
print(f"Tempo de execução: {end - start:.2f} segundos")

---
## 8. Comparação Visual dos Resultados

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(original_img)
axes[0].set_title('Imagem Original', fontsize=14)
axes[0].axis('off')

axes[1].imshow(dream_img_mainloop)
axes[1].set_title('DeepDream - Main Loop', fontsize=14)
axes[1].axis('off')

axes[2].imshow(img)
axes[2].set_title('DeepDream - Com Oitavas', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## 9. Explicação dos Resultados

### Imagem Onírica Obtida por Main Loop

O **Main Loop** aplica o algoritmo de subida de gradiente em uma **única escala** da imagem. O processo funciona da seguinte forma:

1. A imagem é passada pelo modelo InceptionV3
2. As ativações das camadas selecionadas (mixed3 e mixed5) são calculadas
3. Os gradientes são computados em relação à imagem de entrada
4. A imagem é modificada para aumentar essas ativações

**Características do resultado:**
- Padrões **locais** e de **granularidade uniforme** são realçados
- Texturas como pelo do gato e cristais de neve são amplificados
- O efeito é mais **sutil** e mantém a estrutura geral da imagem
- Pode apresentar **ruído** devido à operação em uma única escala
- Padrões tendem a ter o **mesmo tamanho** em toda a imagem

---

### Imagem Onírica Obtida ao Levar o Modelo até uma Oitava

O método de **oitavas** (octaves) aplica o DeepDream em **múltiplas escalas** da imagem:

1. A imagem é redimensionada para diferentes tamanhos (oitavas)
2. O DeepDream é aplicado em cada escala, começando pela menor
3. O resultado é propagado para a próxima escala maior
4. O processo se repete até a escala original

**Características do resultado:**
- Padrões aparecem em **diferentes escalas** (pequenos e grandes)
- O efeito é mais **intenso** e **dramático**
- Estruturas **maiores** e mais **organizadas** emergem
- A imagem apresenta maior **coerência visual**
- Formas reconhecíveis (olhos, rostos, animais) podem aparecer devido às camadas mais profundas

---

### Diferenças entre Main Loop e Oitavas

| Aspecto | Main Loop | Com Oitavas |
|---------|-----------|-------------|
| **Escala dos padrões** | Uniforme (única escala) | Múltiplas escalas |
| **Intensidade do efeito** | Mais sutil | Mais dramático |
| **Ruído** | Pode ser mais ruidoso | Menos ruído, mais suave |
| **Estruturas emergentes** | Padrões pequenos e repetitivos | Formas maiores e organizadas |
| **Tempo de processamento** | Mais rápido | Mais lento (múltiplas passagens) |
| **Coerência visual** | Menor | Maior |

**Em resumo:**
- O **Main Loop** é ideal para um efeito mais sutil, realçando texturas e padrões existentes
- As **Oitavas** produzem um resultado mais surreal e onírico, com padrões em múltiplas escalas que tornam a imagem mais "alucinógena"
- A escolha entre os métodos depende do efeito visual desejado: sutil (Main Loop) ou dramático (Oitavas)

---

### Parâmetros Ajustáveis

Você pode alterar os seguintes parâmetros para modificar a aparência da imagem DeepDream:

- **`steps`**: Número de iterações de subida de gradiente (mais passos = efeito mais forte)
- **`step_size`**: Taxa de aprendizado (valores maiores = mudanças mais intensas)
- **`OCTAVE_SCALE`**: Fator de escala entre oitavas (valores maiores = mais variação de escala)
- **`names`** (camadas): Diferentes camadas capturam diferentes níveis de abstração
  - Camadas iniciais: texturas e bordas
  - Camadas intermediárias: padrões e formas
  - Camadas finais: objetos e conceitos de alto nível

---

# Fim do Notebook